# Dimensional — Date Dimension

Build `globalmart.gold.dim_date` for **2016-01-01 through 2020-12-31** with calendar attributes, weekend/month-boundary flags, and fiscal calendar fields (fiscal year starts **April 1**).

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.dimensional.date_dim as date_dim_module
importlib.reload(date_dim_module)

from src.dimensional.date_dim import DateDimConfig, build_date_dimension, run_date_dimension
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = DateDimConfig()
print("Loaded from:", date_dim_module.__file__)

In [ ]:
dim = build_date_dimension(spark, config)
dim.write.format("delta").mode("overwrite").saveAsTable(config.target_table)
print("Rows:", spark.table(config.target_table).count())
display(spark.table(config.target_table).orderBy("date_key").limit(10))

In [ ]:
from pyspark.sql import functions as F

display(
    spark.table(config.target_table)
    .filter((F.col("month") == 4) & (F.col("day_of_month") == 1))
    .select("full_date", "fiscal_year", "fiscal_quarter", "fiscal_month", "year")
    .orderBy("date_key")
)

In [ ]:
import json
from pyspark.sql import functions as F

written = spark.table(config.target_table)
row_count = written.count()

report = {
    "task": "dim_date",
    "target_table": config.target_table,
    "date_range": {"start": config.start_date, "end": config.end_date},
    "fiscal_year_start_month": config.fiscal_year_start_month,
    "row_count": row_count,
    "expected_row_count": 1827,
    "row_count_matches_expected": row_count == 1827,
    "weekend_days": written.filter(F.col("is_weekend")).count(),
    "fiscal_april_samples": [
        row.asDict()
        for row in written.filter((F.col("month") == 4) & (F.col("day_of_month") == 1))
        .orderBy("date_key")
        .limit(5)
        .collect()
    ],
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/dimensional_date_dim.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)